# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library following the Croissant schema.

### Dataset Source
The dataset source is provided via this Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access and display the dataset metadata (using properties, not subscripting)
print(f"Dataset: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id` references. This will help you identify the data structure, as every entity is referenced by its `@id`.

In [ ]:
# List all record sets and fields by their @id
rs_objs = dataset.metadata.record_sets
print(f"Found {len(rs_objs)} record sets.")
for rs in rs_objs:
    print(f"\nRecordSet name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}\n  Fields:")
    for field in rs.fields:
        col = field.column_id if hasattr(field, 'column_id') else field.id
        print(f"    - Name: {field.name} | Field @id: {field.id} | Column: {col} | Type: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the `@id` values from the overview above.

For this notebook, we'll extract all available record sets for demonstration and use the main one for further analysis.

In [ ]:
# Prepare list of record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

# Extract records to Pandas DataFrames
dataframes = {}
for rs_id in record_set_ids:
    recs = list(dataset.records(record_set=rs_id))
    if recs:
        dataframes[rs_id] = pd.DataFrame(recs)
        print(f"Loaded record set: {rs_id}, shape: {dataframes[rs_id].shape}")
    else:
        print(f"Record set {rs_id} is empty or could not be loaded.")

# Display available DataFrame column info for first non-empty record set
main_rs_id = None
for rs_id in dataframes:
    main_rs_id = rs_id
    break

if main_rs_id:
    print("\nColumns in DataFrame (by @id):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No record sets loaded data.")

## 4. Exploratory Data Analysis (EDA)
Apply typical EDA steps on the selected record set. Here, use only `@id`s for fields, as per the Croissant structure.

We will select a numeric field, filter, normalize, and group as demonstration.

In [ ]:
if main_rs_id:
    df = dataframes[main_rs_id]
    
    # Choose numeric field by @id for filtering and normalization
    # For example, let's try to find a numeric field (e.g., 'Normalized Abundance', 'MW', etc.)
    # You can inspect columns as follows:
    print("Available columns for analysis:")
    print(df.columns.tolist())

    # We'll attempt to pick a likely normalized abundance field id if present
    numeric_field_id = None
    candidate_ids = [
        'NormalizedAbundance_@id',   # Replace with actual field @id as needed
        'cr:NormalizedAbundance',    # Possible pattern
        'cr:MW',                     # Molecular Weight
        'cr:peptide_count',          # Example
        'cr:Score',
        # Add other likely numeric field @ids from printout above
    ]
    for cid in candidate_ids:
        if cid in df.columns:
            numeric_field_id = cid
            break

    if numeric_field_id is None:
        # fallback: pick first numeric-like column
        for c in df.columns:
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_field_id = c
                break

    print(f"\nAnalyzing field: {numeric_field_id}")

    # Filtering for demonstration (if field found and is numeric)
    if numeric_field_id is not None and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping by a categorical field (e.g., 'cr:description', if available)
        group_field_id = None
        for gid in ['cr:description', 'cr:ProteinName', 'cr:accession']:
            if gid in df.columns:
                group_field_id = gid
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (showing first few results):")
            display(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA demonstration.")
else:
    print("No data loaded to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, using field and record set `@id`s where applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if main_rs_id and numeric_field_id and numeric_field_id in dataframes[main_rs_id].columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_rs_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    # If a group_field_id is determined above, a boxplot:
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[main_rs_id])
        plt.xticks(rotation=35)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, you have explored the structure and content of the FAIR² dataset using its Croissant description and the `mlcroissant` Python library. 

- You inspected metadata, record set structure, and available data fields using their Croissant `@id` references.
- You extracted records of interest as Pandas DataFrames.
- You performed basic filtering, normalization, and grouping operations referencing fields by their `@id` to prepare and analyze the experimental protein data.
- You visualized distributions of selected numeric fields, supporting further biological or computational analysis.

For more sophisticated analyses, consider using additional fields, combining several record sets, or integrating this data with other experiments. Refer to the Croissant schema (FAIR²) via the link above for precise documentation of all available record and field `@id`s.